In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/embeddings-lab/exercises")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Exercises 01 · Counts to vectors
Fill in each `YOUR CODE HERE` block. Every task has a self-check cell — run it
to verify. Solutions: `solutions/ex01_solutions.ipynb`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

## Task 1 — implement PPMI
`ppmi(C)[i,j] = max(0, log( p(i,j) / (p(i)·p(j)) ))`, with unseen pairs → 0.

In [ ]:
def ppmi(C):
    # ============ YOUR CODE HERE ============
    raise NotImplementedError("implement me, then re-run")

# self-check: C = [[0,4],[4,0]] → p(0,1)=1/2, p(0)=p(1)=1/2 → PMI = log 2
out = ppmi(np.array([[0.0, 4.0], [4.0, 0.0]]))
assert np.allclose(out, [[0, np.log(2)], [np.log(2), 0]]), out
print("ppmi ✓")

## Task 2 — the SGNS gradient
For one (center `v`, positive `u⁺`, negatives `U⁻` of shape (K,d)) example:
`L = −log σ(v·u⁺) − Σ_k log σ(−v·u⁻_k)`.
Return `(dL/dv, dL/du⁺, dL/dU⁻)`. Hint: both gradients are sigmoid residuals
times the *other* vector.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sgns_grads(v, upos, Uneg):
    # ============ YOUR CODE HERE ============
    raise NotImplementedError("implement me, then re-run")

# self-check: finite differences
def sgns_loss(v, upos, Uneg):
    return -np.log(sigmoid(v @ upos)) - np.log(sigmoid(-Uneg @ v)).sum()

v, upos, Uneg = rng.normal(size=4), rng.normal(size=4), rng.normal(size=(3, 4))
dv, dupos, dUneg = sgns_grads(v, upos, Uneg)
eps = 1e-6
for arr, g, name in [(v, dv, "dv"), (upos, dupos, "dupos"), (Uneg, dUneg, "dUneg")]:
    num = np.zeros_like(arr)
    it = np.nditer(arr, flags=["multi_index"])
    while not it.finished:
        k = it.multi_index
        old = arr[k]
        arr[k] = old + eps; lp = sgns_loss(v, upos, Uneg)
        arr[k] = old - eps; lm = sgns_loss(v, upos, Uneg)
        arr[k] = old
        num[k] = (lp - lm) / (2 * eps)
        it.iternext()
    assert np.abs(num - g).max() < 1e-5, name
print("sgns_grads ✓ (matches finite differences)")

## Task 3 — the analogy function
`analogy(a, b, c)`: nearest word to `a − b + c` by cosine, **excluding** a, b,
c. Uses the vectors saved by notebook 01 (run it first).

In [ ]:
art = np.load("../artifacts/word_vectors.npz", allow_pickle=True)
W = art["W_svd"]; vocab = list(art["vocab"]); w2i = {w: i for i, w in enumerate(vocab)}

def analogy(a, b, c, W=W):
    # ============ YOUR CODE HERE ============
    raise NotImplementedError("implement me, then re-run")

assert analogy("king", "man", "woman") == "queen", analogy("king", "man", "woman")
print("analogy ✓ :", "king − man + woman =", analogy("king", "man", "woman"),
      "| prince − boy + girl =", analogy("prince", "boy", "girl"))

## Task 4 (open) — window size changes what "similar" means
Rebuild notebook 01's co-occurrence with `WINDOW = 1` and `WINDOW = 8` and
compare neighbours of `king`. Small windows → substitutable words (other
people); large windows → topical associates (palace, throne). No assert —
write two sentences on what you observe.